# 00-demand: demand per hex on the slice

How many orders a day we expect from each **hex**, with every factor visible:
**demand = households × affluence multiplier × capture share**. This is a model, not a forecast.
Population is from WorldPop, night-time light from NASA Black Marble (VIIRS). Overture places are not used here.

In [ ]:
import h3
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon, box

from catchment.area import load_area
from catchment.fetch import DATA
from catchment.load import worldpop_pixels, viirs_at

FIGS = DATA.parent / "notebooks" / "figures"
FIGS.mkdir(parents=True, exist_ok=True)
area = load_area()

## Assumptions

Every number the model rests on, in one place. **Verified** means checked against the source on the date shown;
**Unverified** means taken from a secondary report, a search summary or memory and still to be checked.

In [ ]:
# --- Assumptions ---------------------------------------------------------
HOUSEHOLD_SIZE = 4.5          # persons per household, one value for every hex
M_LOW, M_HIGH = 0.5, 1.5      # affluence multiplier bounds

# Capture-share calibration inputs
BLINKIT_ORDERS_Q2 = 398.9e6   # Blinkit orders, Q2 FY27 (Jul-Sep 2026)
Q2_DAYS = 92
BLINKIT_STORES = 2751         # at quarter end
NCR_STORES = 400              # dark stores, all brands, NCR
NCR_POPULATION = 34.7e6       # Delhi urban agglomeration, 2025 projection

ORDERS_PER_STORE_DAY = BLINKIT_ORDERS_Q2 / Q2_DAYS / BLINKIT_STORES
NCR_HOUSEHOLDS = NCR_POPULATION / HOUSEHOLD_SIZE
# calibration target: all-brand orders per household per day; the capture share is set from it below,
# once the affluence multipliers are known
ORDERS_PER_HOUSEHOLD_DAY = NCR_STORES * ORDERS_PER_STORE_DAY / NCR_HOUSEHOLDS

assumptions = pd.DataFrame(
    [
        ("Household size", f"{HOUSEHOLD_SIZE} persons", "NFHS-5 (2019-21) Haryana state report, IIPS/MoHFW, Table 4 (urban mean household size). https://dhsprogram.com/pubs/pdf/FR374/FR374_Haryana.pdf", "2026-10-29", "Verified"),
        ("Multiplier bounds", f"{M_LOW} to {M_HIGH}", "No external source; a labelled assumption (decision log, 2026-10-29)", "-", "Assumption"),
        ("Blinkit orders, Q2 FY27", f"{BLINKIT_ORDERS_Q2/1e6:.1f} M over {Q2_DAYS} days", "Eternal Q2 FY27 results as reported by MediaNama, 'Eternal clocks Rs 125 crore profit in Q2FY27', https://www.medianama.com/2026/10/223-eternal-125-crore-profit-q2fy27/ (shareholder letter not checked)", "2026-10-22", "Unverified"),
        ("Blinkit stores", f"{BLINKIT_STORES:,} at quarter end", "Same MediaNama report", "2026-10-22", "Unverified"),
        ("NCR dark stores (all brands)", f"~{NCR_STORES}", "Savills India, report published Dec 2025, counts as of Oct 2025 (seen in a search summary only)", "2025-10", "Unverified"),
        ("NCR population", f"{NCR_POPULATION/1e6:.1f} M", "UN DESA World Urbanization Prospects 2018, Delhi urban agglomeration, 2025 projection (from memory, not checked)", "2018 (proj. 2025)", "Unverified"),
    ],
    columns=["Input", "Value", "Source", "Date", "Status"],
)
pd.set_option("display.max_colwidth", None)
display(assumptions.style.hide(axis="index").set_properties(**{"text-align": "left"}))

print(f"ORDERS_PER_STORE_DAY = {BLINKIT_ORDERS_Q2:,.0f} / {Q2_DAYS} / {BLINKIT_STORES:,} = {ORDERS_PER_STORE_DAY:,.0f} orders per store per day")
print(f"NCR households = {NCR_POPULATION:,.0f} / {HOUSEHOLD_SIZE} = {NCR_HOUSEHOLDS:,.0f}")
print(f"Calibration target = {NCR_STORES} x {ORDERS_PER_STORE_DAY:,.0f} / {NCR_HOUSEHOLDS:,.0f} = {ORDERS_PER_HOUSEHOLD_DAY:.4f} orders per household per day")

The default capture share is calibrated so that an average household places all-brand quick-commerce orders at the
NCR rate: stores × orders per store per day ÷ households. Because demand also multiplies by the affluence multiplier,
the share is that rate divided by the area's household-weighted mean multiplier (set after the multiplier, below), so
the area's total demand matches the calibration. It is the lever's starting value, not a fixed one.

Caveats, all making the figure rough: Blinkit's orders per store stand in for every brand's; the store count (NCR,
Oct 2025) and the population (Delhi urban agglomeration, which takes in Gurugram and Noida but not all of NCR) cover
slightly different ground and dates; the order figure is Jul-Sep 2026, a year after the store count, so the rate is
probably a little high. Later notebooks set store capacity to 1.5 × `ORDERS_PER_STORE_DAY`.


## The slice

A lon/lat box, 77.005-77.105 E and 28.405-28.480 N, cut to the area. It is large enough to hold several 10-minute
catchments (one is about 17 km²), and it contrasts the dense old city (sectors 4-14, Sadar) with the Golf Course Road
and Sohna Road high-rises.

In [ ]:
slice_poly = box(77.005, 28.405, 77.105, 28.480).intersection(area)

def km2(geom):
    return gpd.GeoSeries([geom], crs=4326).to_crs(32643).area.iloc[0] / 1e6

print(f"Slice area: {km2(slice_poly):.1f} km2  (area: {km2(area):.1f} km2)")

## Hexes and households

Each WorldPop pixel (people per 100 m pixel) goes to the res-9 hex containing its centre. Res-9 hexes are about
0.1 km². A hex belongs to the area, or the slice, if its centre lies inside. Res-8 hexes (about 0.74 km²) are summed
from their res-9 children, so totals agree across resolutions; their membership is by their own centre. Population
in pixels whose hex centre falls outside the area is dropped.

In [ ]:
wp = worldpop_pixels()

pix = pd.DataFrame({"hex": [h3.latlng_to_cell(y, x, 9) for x, y in zip(wp.lon, wp.lat)], "pop": wp["pop"]})
pop9 = pix.groupby("hex")["pop"].sum()

# every res-9 hex whose centre is in the area, including those with no population pixels
area9 = sorted(h3.geo_to_cells(area, 9))
hex9 = pd.DataFrame(index=pd.Index(area9, name="hex"))
hex9["pop"] = pop9.reindex(hex9.index).fillna(0.0)
ctr = [h3.cell_to_latlng(c) for c in hex9.index]
hex9["lat"], hex9["lon"] = [c[0] for c in ctr], [c[1] for c in ctr]
hex9["in_slice"] = [slice_poly.contains(Point(x, y)) for x, y in zip(hex9.lon, hex9.lat)]
hex9["households"] = hex9["pop"] / HOUSEHOLD_SIZE
print(f"Population in area hexes: {hex9['pop'].sum():,.0f} (raster total {wp['pop'].sum():,.0f}; the rest is outside the area)")

## Affluence multiplier

Black Marble radiance is sampled at each res-9 hex centre and ranked across **all** area hexes (percentile p, min to 0,
max to 1). The multiplier is `0.5 + p × (1.5 − 0.5)`: the dimmest hexes count half, the brightest one and a half.
Hexes with no radiance value take the minimum. VIIRS pixels (about 460 m) are coarser than res-9 hexes, so
neighbouring hexes share a value. A res-8 multiplier is the household-weighted mean of its children's.

In [ ]:
rad = viirs_at(hex9.lon, hex9.lat)

n_nan = int(np.isnan(rad).sum())
hex9["radiance"] = np.where(np.isnan(rad), np.nanmin(rad), rad)
pct = hex9["radiance"].rank(pct=True, method="average")
hex9["p"] = (pct - pct.min()) / (pct.max() - pct.min())
hex9["multiplier"] = M_LOW + hex9["p"] * (M_HIGH - M_LOW)
print(f"Radiance NaN (filled with min): {n_nan} of {len(hex9)} hexes")
print(f"Radiance == 0: {int((hex9.radiance == 0).sum())} hexes; distinct radiance values: {hex9.radiance.nunique()}")
print(hex9.radiance.describe().round(2).to_string())

MEAN_MULTIPLIER = np.average(hex9["multiplier"], weights=hex9["households"])
CAPTURE_SHARE = ORDERS_PER_HOUSEHOLD_DAY / MEAN_MULTIPLIER
print(f"Household-weighted mean multiplier (area) = {MEAN_MULTIPLIER:.3f}")
print(f"CAPTURE_SHARE (default) = {ORDERS_PER_HOUSEHOLD_DAY:.4f} / {MEAN_MULTIPLIER:.3f} = {CAPTURE_SHARE:.4f}")

## Demand

Orders per day = households × affluence multiplier × capture share, at res 9, then res 8 as the sum of children.

In [ ]:
hex9["demand"] = hex9["households"] * hex9["multiplier"] * CAPTURE_SHARE

hex9["parent"] = [h3.cell_to_parent(c, 8) for c in hex9.index]
def wmean(g):
    w = g["households"]
    return np.average(g["multiplier"], weights=w) if w.sum() > 0 else g["multiplier"].mean()
hex8 = hex9.groupby("parent").agg(pop=("pop", "sum"), households=("households", "sum"), demand=("demand", "sum"))
hex8["multiplier"] = hex9.groupby("parent").apply(wmean, include_groups=False)
hex8.index.name = "hex"
ctr8 = [h3.cell_to_latlng(c) for c in hex8.index]
hex8["in_area"] = [area.contains(Point(c[1], c[0])) for c in ctr8]
hex8["in_slice"] = [slice_poly.contains(Point(c[1], c[0])) for c in ctr8]
hex8a = hex8[hex8.in_area]   # res-8 hexes whose centre is in the area

In [ ]:
s9, s8 = hex9[hex9.in_slice], hex8a[hex8a.in_slice]
report = pd.DataFrame(
    {
        "res 9": [len(hex9), len(s9), s9.households.sum(), s9.demand.sum(), hex9.households.sum(), hex9.demand.sum()],
        "res 8": [len(hex8a), len(s8), s8.households.sum(), s8.demand.sum(), hex8.households.sum(), hex8.demand.sum()],
    },
    index=["hexes, area", "hexes, slice", "households, slice", "demand, slice (orders/day)", "households, area", "demand, area (orders/day)"],
)
display(report.map(lambda v: f"{v:,.0f}"))
print(f"Slice demand / ORDERS_PER_STORE_DAY = {s9.demand.sum() / ORDERS_PER_STORE_DAY:.1f} stores' worth of orders at ORDERS_PER_STORE_DAY (res 9)")
print(f"Area demand / ORDERS_PER_STORE_DAY = {hex9.demand.sum() / ORDERS_PER_STORE_DAY:.1f} stores' worth of orders at ORDERS_PER_STORE_DAY (res 9)")

Results from the run on 2026-10-29 (copied here because outputs are stripped on commit; re-running updates the
printed table, not this one):

| | res 9 | res 8 |
|---|---:|---:|
| hexes, area | 4,859 | 697 |
| hexes, slice | 713 | 102 |
| households, slice | 183,481 | 181,696 |
| **demand, slice (orders/day)** | **16,523** | **16,378** |
| households, area | 955,163 | 955,163 |
| demand, area (orders/day) | 78,092 | 78,092 |

Slice: 77.5 km². Calibration target 0.082 orders per household per day; area mean multiplier 1.167; default capture
share 0.070; `ORDERS_PER_STORE_DAY` ≈ 1,576. Sanity check: the area's demand is about 50 stores' worth of orders,
about 12% of NCR's ~400 stores, in line with its share of NCR households (955 k of 7.7 M). That agreement holds by
construction of the calibration; it checks the arithmetic, not the inputs. The slice is about 10.5 stores' worth.


Res-9 and res-8 slice totals differ by about 1% at the slice edge, where a res-8 hex's centre is in the slice but some
children's centres are not, or the other way round. Area totals agree exactly, because every res-8 hex is summed from area res-9 hexes (so the res-8 area hex count, by centre, is lower than the number of res-8 hexes summed).

In [ ]:
def boundary(c):
    return Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(c)])

def to_gdf(df):
    return gpd.GeoDataFrame(df.copy(), geometry=[boundary(c) for c in df.index], crs=4326)

g9, g8 = to_gdf(s9), to_gdf(s8)
g8_area = to_gdf(hex8a)
slice_gs = gpd.GeoSeries([slice_poly], crs=4326)
area_gs = gpd.GeoSeries([area], crs=4326)

def two_panel(col, label, title, fname, cmap="viridis"):
    fig, axes = plt.subplots(1, 2, figsize=(12, 6.5), constrained_layout=True)
    for ax, g, res in zip(axes, (g9, g8), (9, 8)):
        g.plot(column=col, ax=ax, cmap=cmap, linewidth=0.1, edgecolor="white")
        slice_gs.boundary.plot(ax=ax, color="black", linewidth=1)
        ax.set_title(f"H3 res {res} ({len(g)} hexes)")
        ax.set_aspect(1 / np.cos(np.radians(28.44)))
        ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=g[col].min(), vmax=g[col].max()))
        fig.colorbar(sm, ax=ax, shrink=0.8, label=label)
    fig.suptitle(title)
    fig.savefig(FIGS / fname, dpi=120)
    plt.show()

## Figures

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8), constrained_layout=True)
g8_area.plot(column="households", ax=ax, cmap="viridis", linewidth=0, legend=True,
             legend_kwds={"label": "households per res-8 hex", "shrink": 0.7})
area_gs.boundary.plot(ax=ax, color="grey", linewidth=0.8)
slice_gs.boundary.plot(ax=ax, color="red", linewidth=2)
ax.set_aspect(1 / np.cos(np.radians(28.5)))
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title(f"The slice (red, {km2(slice_poly):.0f} km2) inside the area (grey, {km2(area):.0f} km2)")
fig.text(0.5, -0.01, "Households per res-8 hex across the area. The slice runs from old Gurugram to Golf Course Road / Sohna Road.",
         ha="center", fontsize=9)
fig.savefig(FIGS / "00-slice.png", dpi=120, bbox_inches="tight")
plt.show()

![The slice inside the area](figures/00-slice.png)

In [ ]:
two_panel("households", "households per hex", f"Households (population / {HOUSEHOLD_SIZE}), slice", "00-households.png")

![Households per hex, res 9 and res 8](figures/00-households.png)

In [ ]:
two_panel("multiplier", "affluence multiplier", f"Affluence multiplier ({M_LOW} to {M_HIGH}), slice", "00-affluence.png", cmap="magma")

![Affluence multiplier, res 9 and res 8](figures/00-affluence.png)

In [ ]:
two_panel("demand", "orders per day per hex", f"Demand at default capture share {CAPTURE_SHARE:.3f}, slice", "00-demand.png", cmap="magma")

![Demand per hex, res 9 and res 8](figures/00-demand.png)

Colour scales differ between panels because a res-8 hex holds about seven res-9 hexes. Demand is a model: move the
capture share and it scales, move the multiplier bounds and it re-shapes.